# Project Portfolio: Maximizing NYC Taxi Driver Earnings & Fare Optimization

## Executive Summary
In the competitive and fast-paced ride-hailing sector, optimizing revenue generation is crucial for driver retention and platform sustainability. This end-to-end data science project analyzes the factors influencing NYC yellow taxi base fares using programmatic API pipelines, robust exploratory data analysis (EDA), A/B statistical testing, and machine learning models.


# Maximizing Driver Earnings: Payment Method Analysis in the Taxi Industry

## Problem Statement
In the competitive and fast-paced ride-hailing sector, optimizing revenue generation is crucial for sustainable business growth and driver retention. To address this challenge, this project leverages data-driven insights to uncover opportunities for boosting taxi drivers' income streams. Specifically, we investigate whether the chosen payment method significantly influences total fare amounts.

### Core Objectives
1. **Data Ingestion**: Programmatically retrieve real-time operational NYC taxi trip data.
2. **A/B Testing**: Determine if there is a statistically significant difference in base fares between credit card and cash transactions.
3. **Predictive Modeling**: Deploy regression models to estimate base fare amounts from pre-trip attributes.
4. **Operational Interpretation**: Reconstruct physical-scale pricing parameters from standardized model coefficients.

## Research Question
- Does a statistically significant relationship exist between the payment method used and the total fare amount?
- Furthermore, can platforms subtly guide passengers toward revenue-maximizing payment options without compromising user satisfaction?

## Key Libraries & Tools
- **Python**: Core programming and data pipelines
- **Pandas & NumPy**: Data cleaning, transformation, and aggregations
- **Matplotlib & Seaborn**: Exploratory data analysis (EDA) and fare distribution visualization
- **SciPy**: Statistical hypothesis testing (Welch's $t$-test)
- **Scikit-learn**: Standardization, Linear Regression and Random Forest modeling

### Data overview

| Field Name | Type / Format | Description |
| :--- | :--- | :--- |
| **`ID`** | Unique Identifier | Distinct tracking ID assigned to each individual trip record. |
| **`VendorID`** | Categorical Code | Technology service provider that logged the ride:<br>• `1` = Creative Mobile Technologies, LLC<br>• `2` = VeriFone Inc. |
| **`tpep_pickup_datetime`** | Timestamp (`YYYY-MM-DD HH:MM:SS`) | Timestamp recorded when the taximeter was officially engaged at pickup. |
| **`tpep_dropoff_datetime`** | Timestamp (`YYYY-MM-DD HH:MM:SS`) | Timestamp recorded when the taximeter was disengaged at drop-off. |
| **`Passenger_count`** | Integer | Number of passengers aboard the vehicle (manually entered by the driver). |
| **`Trip_distance`** | Float (Miles) | Total distance traveled during the ride as measured by the taximeter. |
| **`PULocationID`** | TLC Taxi Zone ID | NYC TLC Taxi Zone integer identifier where the trip began. |
| **`DOLocationID`** | TLC Taxi Zone ID | NYC TLC Taxi Zone integer identifier where the trip concluded. |
| **`RateCodeID`** | Categorical Code | Fare structure applied at the conclusion of the trip:<br>• `1` = Standard Rate<br>• `2` = JFK Airport<br>• `3` = Newark Airport<br>• `4` = Nassau / Westchester County<br>• `5` = Negotiated Fare<br>• `6` = Group Ride |
| **`Store_and_fwd_flag`** | Binary Flag (`Y` / `N`) | Indicates whether trip data was temporarily cached in onboard vehicle memory due to lack of server connectivity before transmission (`Y` = Stored & Forwarded; `N` = Real-time transmission). |
| **`Payment_type`** | Categorical Code | Method used by the customer to settle the fare:<br>• `1` = Credit Card<br>• `2` = Cash<br>• `3` = No Charge<br>• `4` = Dispute<br>• `5` = Unknown<br>• `6` = Voided Trip |
| **`Fare_amount`** | Float (USD) | Base metered fare computed from time elapsed and distance covered. |
| **`Extra`** | Float (USD) | Mandated supplementary charges (e.g., $0.50 overnight and $1.00 rush-hour surcharges). |
| **`MTA_tax`** | Float (USD) | Flat $0.50 Metropolitan Transportation Authority (MTA) state tax automatically triggered by metered rate rules. |
| **`Improvement_surcharge`** | Float (USD) | Flat $0.30 regulatory improvement surcharge levied at flag drop. |
| **`Tip_amount`** | Float (USD) | Gratuity added to the transaction. *Note: Populated exclusively for credit card payments; cash tips are unrecorded.* |
| **`Tolls_amount`** | Float (USD) | Cumulative cost of bridge, tunnel, and highway tolls incurred throughout the journey. |
| **`Total_amount`** | Float (USD) | Final sum charged to the passenger across all components (excludes unrecorded cash tips). |

##Setting up google drive for colab to saved data fetched from API

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
from google.colab import drive

drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/Coursera/projects_github/nyc_taxi_2023_v2')

from src.colab_setup import setup_environment
setup_environment()

# Now all your other src modules work cleanly:
from src.api import *
from src.features import *

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Successfully changed directory to: /content/drive/MyDrive/Coursera/projects_github/nyc_taxi_2023_v2
Environment ready. Project root: /content/drive/MyDrive/Coursera/projects_github/nyc_taxi_2023_v2


In [3]:
#Set path to save the fetched dataframe
path = RAW_DIR

In [4]:
# Fetch the 50,000 records (same SoQL query as before, now in src/api.py)
df = fetch_taxi_data(limit=50000)

Fetching records from NYC OpenData API...
Loaded 50,000 rows with 17 columns.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   vendorid               50000 non-null  object        
 1   tpep_pickup_datetime   50000 non-null  datetime64[ns]
 2   tpep_dropoff_datetime  50000 non-null  datetime64[ns]
 3   passenger_count        50000 non-null  float64       
 4   trip_distance          50000 non-null  float64       
 5   ratecodeid             50000 non-null  float64       
 6   store_and_fwd_flag     50000 non-null  object        
 7   pulocationid           50000 non-null  int64         
 8   dolocationid           50000 non-null  int64         
 9   payment_type           50000 non-null  int64         
 10  fare_amount            50000 non-null  float64       
 11  extra                  50000 non-null  f

In [6]:
# Save the dataframe as CSV in data/raw/
save_dataframe(df, path)

DataFrame successfully saved to: /content/drive/MyDrive/Coursera/projects_github/nyc_taxi_2023_v2/data/raw/nyc_yellow_taxi_data_2023.csv


PosixPath('/content/drive/MyDrive/Coursera/projects_github/nyc_taxi_2023_v2/data/raw/nyc_yellow_taxi_data_2023.csv')